In [6]:
import pandas as pd
import numpy as np

master = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/user_day_master.csv")
master['day'] = pd.to_datetime(master['day'])

logon = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/logon.csv")
logon['date'] = pd.to_datetime(logon['date'])

device = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/device.csv")
device['date'] = pd.to_datetime(device['date'])

answers = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/answers_r42.csv")
malicious_users = answers['user'].unique()

### Feature — after-hours logon flag/count
Based on EDA, off-hours (before 6 AM or after 7 PM) is a real signal

In [3]:
logon['hour'] = logon['date'].dt.hour
logon['day'] = logon['date'].dt.date
logon['is_after_hours'] = (logon['hour'] < 6) | (logon['hour'] >= 19)

after_hours_daily = logon.groupby(['user','day'])['is_after_hours'].sum().reset_index(name='after_hours_logon_count')
after_hours_daily['day'] = pd.to_datetime(after_hours_daily['day'])

print(after_hours_daily.head())

      user        day  after_hours_logon_count
0  AAE0190 2010-01-04                        0
1  AAE0190 2010-01-05                        0
2  AAE0190 2010-01-06                        0
3  AAE0190 2010-01-07                        0
4  AAE0190 2010-01-08                        0


### Feature — first-time USB usage flag

In [4]:
device['day'] = pd.to_datetime(device['date'].dt.date)
device_daily = device.groupby(['user','day']).size().reset_index(name='device_count')

# For each user, find their first-ever USB use date
first_usb_date = device_daily.groupby('user')['day'].min().reset_index(name='first_usb_date')

device_daily = device_daily.merge(first_usb_date, on='user', how='left')
device_daily['is_first_usb_use'] = device_daily['day'] == device_daily['first_usb_date']

print(device_daily.head())

      user        day  device_count first_usb_date  is_first_usb_use
0  AAF0535 2010-01-05             4     2010-01-05              True
1  AAF0535 2010-01-06             2     2010-01-05             False
2  AAF0535 2010-01-07             4     2010-01-05             False
3  AAF0535 2010-01-08             8     2010-01-05             False
4  AAF0535 2010-01-11             2     2010-01-05             False


### Feature — rolling personal baseline (deviation from own history)

In [9]:
master = master.sort_values(['user','day'])

for col in ['logon_count','device_count','file_count','email_count','http_count']:
    master[f'{col}_7day_avg'] = (
        master.groupby('user')[col]
        .transform(lambda x: x.shift(1).rolling(window=7, min_periods=3).mean())
    )
    master[f'{col}_deviation'] = master[col] - master[f'{col}_7day_avg']

print(master[['user','day','device_count','device_count_7day_avg','device_count_deviation']].dropna().head(10))

       user        day  device_count  device_count_7day_avg  \
3   AAE0190 2010-01-07           0.0                    0.0   
4   AAE0190 2010-01-08           0.0                    0.0   
5   AAE0190 2010-01-11           0.0                    0.0   
6   AAE0190 2010-01-12           0.0                    0.0   
7   AAE0190 2010-01-13           0.0                    0.0   
8   AAE0190 2010-01-14           0.0                    0.0   
9   AAE0190 2010-01-15           0.0                    0.0   
10  AAE0190 2010-01-18           0.0                    0.0   
11  AAE0190 2010-01-19           0.0                    0.0   
12  AAE0190 2010-01-20           0.0                    0.0   

    device_count_deviation  
3                      0.0  
4                      0.0  
5                      0.0  
6                      0.0  
7                      0.0  
8                      0.0  
9                      0.0  
10                     0.0  
11                     0.0  
12              

### Observation:
After-hours logon count: AAE0190 shows 0 every day — makes sense, this is a normal user with a rigid 9-9 schedule (same user from our earlier http.csv investigation), so zero after-hours activity is expected and correct.
First-time USB flag: Working exactly as designed — AAF0535's very first USB day (2010-01-05) correctly shows True, every day after shows False.
AAE0190 shows 0.0 for device_count, device_count_7day_avg, AND device_count_deviation across every row. This isn't a bug — it's because this particular user never uses USB devices at all, so there's no baseline to deviate from (0 minus 0 average = 0 deviation). This is the expected, correct behavior for a genuinely "boring" normal user.

### verify with actual malicious user — check AAM0658

In [10]:
suspect = master[master['user'] == 'AAM0658'][['user','day','device_count','device_count_7day_avg','device_count_deviation']]
suspect = suspect[suspect['day'] >= pd.Timestamp('2010-10-15')]
print(suspect.head(20))

         user        day  device_count  device_count_7day_avg  \
1402  AAM0658 2010-10-15           0.0               0.000000   
1403  AAM0658 2010-10-18           0.0               0.000000   
1404  AAM0658 2010-10-19           0.0               0.000000   
1405  AAM0658 2010-10-20           0.0               0.000000   
1406  AAM0658 2010-10-21           6.0               0.000000   
1407  AAM0658 2010-10-22           0.0               0.857143   
1408  AAM0658 2010-10-23           2.0               0.857143   
1409  AAM0658 2010-10-25           0.0               1.142857   
1410  AAM0658 2010-10-26           0.0               1.142857   
1411  AAM0658 2010-10-27           2.0               1.142857   
1412  AAM0658 2010-10-28           0.0               1.428571   
1413  AAM0658 2010-10-29           2.0               1.428571   
1414  AAM0658 2010-11-01           0.0               0.857143   
1415  AAM0658 2010-11-02           1.0               0.857143   
1416  AAM0658 2010-11-03 

### Feature Engineering Note: Rolling Baseline Absorption Effect
Testing the device_count_deviation feature on user AAM0658 (Scenario 1) 
shows a large deviation spike (+6.0) on the first anomalous day (Oct 21), 
but subsequent deviations shrink and even turn negative as the 7-day 
rolling average absorbs the new (malicious) behavior into its baseline. 
This confirms rolling-window features are most sensitive to the ONSET of 
anomalous behavior rather than its persistence, a known limitation that 
motivates using a longer, stable-period autoencoder model (trained only 
on the pre-June 2010 "clean" period) alongside rolling features, rather 
than relying on rolling deviation alone.

### Feature — Email to external domains
This is a key signal per Scenario 1 (uploads to wikileaks.org) and general exfiltration behavior.

In [19]:
email = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/email.csv", usecols=['date','user','to','cc','bcc','from'])
email['date'] = pd.to_datetime(email['date'])
email['day'] = email['date'].dt.date
print(email.head())

                 date     user  \
0 2010-01-02 07:11:45  LAP0338   
1 2010-01-02 07:12:16  MOH0273   
2 2010-01-02 07:13:00  LAP0338   
3 2010-01-02 07:13:17  LAP0338   
4 2010-01-02 07:13:28  MOH0273   

                                                  to  \
0  Dean.Flynn.Hines@dtaa.com;Wade_Harrison@lockhe...   
1                        Odonnell-Gage@bellsouth.net   
2                         Penelope_Colon@netzero.com   
3                          Judith_Hayden@comcast.net   
4  Bond-Raymond@verizon.net;Alea_Ferrell@msn.com;...   

                                cc                          bcc  \
0  Nathaniel.Hunter.Heath@dtaa.com                          NaN   
1                              NaN                          NaN   
2                              NaN                          NaN   
3                              NaN                          NaN   
4                              NaN  Odonnell-Gage@bellsouth.net   

                         from         day  
0   Lynn.Ad

### Now flag emails going to addresses outside the company domain. First, check what a company email looks like:

In [18]:
print(email['from'].head(10))

0          Lynn.Adena.Pratt@dtaa.com
1                MOH68@optonline.net
2         Lynn_A_Pratt@earthlink.net
3         Lynn_A_Pratt@earthlink.net
4                MOH68@optonline.net
5          Hollee_Becker@hotmail.com
6    Noelani.W.Kennedy@optonline.net
7     Libby.Rosalyn.Richard@dtaa.com
8     Libby.Rosalyn.Richard@dtaa.com
9     Libby.Rosalyn.Richard@dtaa.com
Name: from, dtype: object


In [20]:
company_domain = "dtaa.com" 

def is_external(addr_list):
    if pd.isna(addr_list):
        return False
    addresses = str(addr_list).split(';')
    return any(company_domain not in a for a in addresses)

email['to_external'] = email['to'].apply(is_external)

external_email_daily = email.groupby(['user','day'])['to_external'].sum().reset_index(name='external_email_count')
external_email_daily['day'] = pd.to_datetime(external_email_daily['day'])

print(external_email_daily.head())

      user        day  external_email_count
0  AAE0190 2010-01-04                     2
1  AAE0190 2010-01-05                     7
2  AAE0190 2010-01-06                     6
3  AAE0190 2010-01-07                     5
4  AAE0190 2010-01-08                     2


### Merge every engineered feature into your master table

In [21]:
master = master.merge(after_hours_daily, on=['user','day'], how='left')
master = master.merge(device_daily[['user','day','is_first_usb_use']], on=['user','day'], how='left')
master = master.merge(external_email_daily, on=['user','day'], how='left')

# Fill missing values (days with no activity in a source = 0)
master['after_hours_logon_count'] = master['after_hours_logon_count'].fillna(0)
master['is_first_usb_use'] = master['is_first_usb_use'].fillna(False)
master['external_email_count'] = master['external_email_count'].fillna(0)

print(master.shape)
print(master.columns.tolist())

(330452, 20)
['user', 'day', 'logon_count', 'device_count', 'file_count', 'email_count', 'http_count', 'logon_count_7day_avg', 'logon_count_deviation', 'device_count_7day_avg', 'device_count_deviation', 'file_count_7day_avg', 'file_count_deviation', 'email_count_7day_avg', 'email_count_deviation', 'http_count_7day_avg', 'http_count_deviation', 'after_hours_logon_count', 'is_first_usb_use', 'external_email_count']


/var/folders/4r/66t4xbm16h9g6zs8_1r452t00000gq/T/ipykernel_33210/298173308.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master['is_first_usb_use'] = master['is_first_usb_use'].fillna(False)


### Add a simple binary label column (for evaluation later — NOT used in training)

This isn't a feature for the model to learn from directly (remember, this is meant to be an unsupervised/semi-supervised anomaly detection task), but you need it later to evaluate your model's outputs against ground truth:

In [23]:
import pandas as pd

# ---------------------------------------------------
# Convert date columns to datetime
# ---------------------------------------------------
answers['start'] = pd.to_datetime(answers['start'], errors='coerce').dt.normalize()
answers['end'] = pd.to_datetime(answers['end'], errors='coerce').dt.normalize()
master['day'] = pd.to_datetime(master['day'], errors='coerce').dt.normalize()

# ---------------------------------------------------
# Mark malicious users
# ---------------------------------------------------
malicious_users = answers['user'].unique()
master['is_malicious_user'] = master['user'].isin(malicious_users)

# ---------------------------------------------------
# Build lookup table
# ---------------------------------------------------
answers_indexed = answers.set_index('user')[['start', 'end']]

# ---------------------------------------------------
# Function to determine if a row falls within
# one of the user's malicious scenario windows
# ---------------------------------------------------
def is_scenario_day(row):

    user = row['user']

    # User has no malicious scenario
    if user not in answers_indexed.index:
        return False

    windows = answers_indexed.loc[user]

    # Handle single scenario
    if isinstance(windows, pd.Series):
        return (
            windows['start'] <= row['day'] <= windows['end']
        )

    # Handle multiple scenarios
    for _, window in windows.iterrows():
        if window['start'] <= row['day'] <= window['end']:
            return True

    return False

# ---------------------------------------------------
# Apply function
# ---------------------------------------------------
master['is_scenario_day'] = master.apply(is_scenario_day, axis=1)

print(
    f"{master['is_scenario_day'].sum()} user-days fall within an actual scenario window"
)

# Optional: preview
print(master[['user', 'day', 'is_malicious_user', 'is_scenario_day']].head())

1364 user-days fall within an actual scenario window
      user        day  is_malicious_user  is_scenario_day
0  AAE0190 2010-01-04              False            False
1  AAE0190 2010-01-05              False            False
2  AAE0190 2010-01-06              False            False
3  AAE0190 2010-01-07              False            False
4  AAE0190 2010-01-08              False            False


In [24]:
master.to_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/final_features.csv", index=False)
print("Saved:", master.shape)

Saved: (330452, 22)


## Feature Engineering Summary
Final feature table (`final_features.csv`) built at user-day granularity, 
combining:
- Raw daily counts: logon, device, file, email, http
- Behavioral features: after_hours_logon_count, is_first_usb_use, 
  external_email_count
- Personal baseline deviations: 7-day rolling average and deviation for 
  each raw count feature
- Ground truth labels (for evaluation only, not model input): 
  is_malicious_user, is_scenario_day

Total: [X] user-day records, [Y] flagged as within an actual scenario window.